## Silver Layer – Insurance Data Cleaning and Standardization

This notebook implements the **Silver layer transformation for the insurance dataset** within the healthcare data platform.

The goal of this stage is to convert the raw insurance data ingested in the Bronze layer into a **clean, standardized, and structured dataset** that can support reliable analytics and joins with other core entities in the platform.

### Transformations performed
- Cleaning and normalization of **insurance identifiers** and payer names
- Standardization of **plan_type** values
- Normalization of **medication and disease references** for downstream joins
- Parsing of raw benefit fields such as **copay** and **coinsurance**
- Conversion of authorization indicators into **boolean flags**
- Standardization of **coverage status**

### Output
The resulting table **`capstone.silver.insurance`** provides a structured dataset that can be joined with:

- `silver.medications` for **medication coverage analysis**
- `silver.diseases` for **disease treatment coverage analysis**

This dataset will support downstream **Gold layer analytics**, including insurance coverage insights, plan comparisons, and medication access analysis.

In [0]:
%python
from pyspark.sql.functions import *

spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

df = spark.table("capstone.bronze.insurance")

silver = (
    df
    # Clean identifiers
    .withColumn("insurance_id", upper(trim(col("insurance_id_source"))))
    .withColumn("insurance_name", initcap(trim(col("insurance_name"))))

    # Keep plan_type simple
    .withColumn("plan_type_base", lower(trim(col("plan_type"))))
    .withColumn(
        "plan_type",
        when(col("plan_type_base").isin("ppo"), "PPO")
        .when(col("plan_type_base").isin("hmo"), "HMO")
        .when(col("plan_type_base").isin("epo"), "EPO")
        .when(col("plan_type_base").isin("pos"), "POS")
        .when(col("plan_type_base").isin("medicaid"), "Medicaid")
        .when(col("plan_type_base").isin("medicare advantage"), "Medicare Advantage")
        .when(col("plan_type_base").isin("government", "govt"), "Government")
        .when(col("plan_type_base").isin("private", "commercial"), "Private")
        .otherwise(initcap(col("plan_type_base")))
    )

    # Clean references
    .withColumn(
        "medication_id_reference",
        upper(trim(regexp_replace(col("medication_id_reference_raw"), "-OLD", "")))
    )
    .withColumn("medication_name_reference", initcap(trim(col("medication_name_reference_raw"))))
    .withColumn("disease_name_reference", initcap(trim(col("disease_name_reference_raw"))))

    # Parse copay
    .withColumn(
        "copay_match",
        regexp_extract(lower(trim(col("copay_raw"))), r"([0-9]+(?:\.[0-9]+)?)", 1)
    )
    .withColumn(
        "copay",
        when(col("copay_match") == "", None).otherwise(col("copay_match").cast("double"))
    )

    # Parse coinsurance
    .withColumn(
        "coinsurance_match",
        regexp_extract(lower(trim(col("coinsurance_raw"))), r"([0-9]+(?:\.[0-9]+)?)", 1)
    )
    .withColumn(
        "coinsurance",
        when(col("coinsurance_match") == "", None).otherwise(col("coinsurance_match").cast("double"))
    )

    # Boolean flags
    .withColumn(
        "requires_prior_auth",
        when(lower(trim(col("prior_auth_required_raw"))).isin("y", "yes", "true", "1", "prior auth", "pa req"), True)
        .when(lower(trim(col("prior_auth_required_raw"))).isin("n", "no", "false", "0"), False)
    )
    .withColumn(
        "requires_step_therapy",
        when(lower(trim(col("step_therapy_raw"))).isin("y", "yes", "true", "1", "step req"), True)
        .when(lower(trim(col("step_therapy_raw"))).isin("n", "no", "false", "0"), False)
    )
    .withColumn(
        "is_specialty_drug",
        when(lower(trim(col("specialty_drug_flag_raw"))).isin("y", "yes", "true", "1", "specialty"), True)
        .when(lower(trim(col("specialty_drug_flag_raw"))).isin("n", "no", "false", "0"), False)
    )

    # Light coverage normalization
    .withColumn("coverage_status_base", lower(trim(col("coverage_status_raw"))))
    .withColumn(
        "coverage_status",
        when(col("coverage_status_base").rlike("covered"), "covered")
        .when(col("coverage_status_base").rlike("not covered"), "not_covered")
        .when(col("coverage_status_base").rlike("excluded"), "excluded")
        .when(col("coverage_status_base").rlike("restricted"), "restricted")
        .otherwise(None)
    )
)

final_df = silver.select(
    "insurance_id",
    "insurance_name",
    "plan_type",
    "medication_id_reference",
    "medication_name_reference",
    "disease_name_reference",
    "copay",
    "coinsurance",
    "requires_prior_auth",
    "requires_step_therapy",
    "is_specialty_drug",
    "coverage_status",
    "source_system",
    "source_record_id"
)

spark.sql("DROP TABLE IF EXISTS capstone.silver.insurance")

(
    final_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("capstone.silver.insurance")
)

print("Loaded: capstone.silver.insurance")
print(final_df.dtypes)